# Experiment 4: Smart Recommendation and Insight Dashboard System

This notebook covers the end-to-end implementation of a movie recommendation system using the MovieLens 100k dataset. 
It includes:
1. Data Loading and Preprocessing
2. Exploratory Data Analysis (EDA)
3. Neural Collaborative Filtering (NCF) Model Design
4. Model Training and Evaluation
5. Saving the Model for Deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout, Multiply
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
import os
import json

# Setting directories
DATA_DIR = "../Data/ml-100k/ml-100k"
OUTPUT_DIR = "../data/processed"
MODEL_DIR = "../models"

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
if not os.path.exists(MODEL_DIR): os.makedirs(MODEL_DIR)

## 1. Data Loading

In [ ]:
def load_data(data_dir):
    # Load Ratings
    ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
    ratings = pd.read_csv(os.path.join(data_dir, 'u.data'), sep='\t', names=ratings_cols, encoding='latin-1')
    
    # Load Movies
    items_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure',
                  'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
                  'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    items = pd.read_csv(os.path.join(data_dir, 'u.item'), sep='|', names=items_cols, encoding='latin-1')
    
    # Load Users
    users_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
    users = pd.read_csv(os.path.join(data_dir, 'u.user'), sep='|', names=users_cols, encoding='latin-1')
    
    return ratings, items, users

ratings, items, users = load_data(DATA_DIR)
print(f"Ratings shape: {ratings.shape}")
ratings.head()

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x='rating', data=ratings, palette='viridis')
plt.title('Distribution of Ratings')
plt.show()

## 3. Preprocessing

In [ ]:
# Indexing users and movies
user_ids = ratings['user_id'].unique().tolist()
movie_ids = ratings['movie_id'].unique().tolist()

user_map = {id: i for i, id in enumerate(user_ids)}
movie_map = {id: i for i, id in enumerate(movie_ids)}

ratings['user_idx'] = ratings['user_id'].map(user_map)
ratings['movie_idx'] = ratings['movie_id'].map(movie_map)

num_users = len(user_map)
num_movies = len(movie_map)

print(f"Number of Users: {num_users}, Number of Movies: {num_movies}")

# Train/Test Split
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# Save processed data for API
ratings.to_csv(os.path.join(OUTPUT_DIR, 'processed_ratings.csv'), index=False)
metadata = {
    'num_users': num_users,
    'num_movies': num_movies,
    'user_map': {int(k): int(v) for k, v in user_map.items()},
    'movie_map': {int(k): int(v) for k, v in movie_map.items()}
}
with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f)

## 4. Modeling (Neural Collaborative Filtering)

In [ ]:
def create_ncf_model(num_users, num_items, embedding_dim=16, layers=[64, 32, 16, 8]):
    user_input = Input(shape=(1,), name='user_input')
    item_input = Input(shape=(1,), name='item_input')

    # GMF part
    user_embedding_gmf = Embedding(num_users, embedding_dim, name='user_embedding_gmf')(user_input)
    item_embedding_gmf = Embedding(num_items, embedding_dim, name='item_embedding_gmf')(item_input)
    gmf_user_flat = Flatten()(user_embedding_gmf)
    gmf_item_flat = Flatten()(item_embedding_gmf)
    gmf_output = Multiply()([gmf_user_flat, gmf_item_flat])

    # MLP part
    user_embedding_mlp = Embedding(num_users, embedding_dim, name='user_embedding_mlp')(user_input)
    item_embedding_mlp = Embedding(num_items, embedding_dim, name='item_embedding_mlp')(item_input)
    mlp_user_flat = Flatten()(user_embedding_mlp)
    mlp_item_flat = Flatten()(item_embedding_mlp)
    mlp_output = Concatenate()([mlp_user_flat, mlp_item_flat])
    
    for i, nodes in enumerate(layers):
        mlp_output = Dense(nodes, activation='relu')(mlp_output)
        mlp_output = Dropout(0.2)(mlp_output)

    # Combine GMF and MLP
    combined = Concatenate()([gmf_output, mlp_output])
    output = Dense(1, activation='linear')(combined)

    model = Model(inputs=[user_input, item_input], outputs=output)
    return model

model = create_ncf_model(num_users, num_movies)
model.compile(optimizer='adam', loss='mse', metrics=['mae', tf.keras.metrics.RootMeanSquaredError()])
model.summary()

## 5. Training

In [ ]:
X_train = [train['user_idx'].values, train['movie_idx'].values]
y_train = train['rating'].values

X_test = [test['user_idx'].values, test['movie_idx'].values]
y_test = test['rating'].values

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=256
)

## 6. Evaluation

In [ ]:
loss, mae, rmse = model.evaluate(X_test, y_test)
print(f"Test RMSE: {rmse:.4f}")

## 7. Save Model

In [ ]:
model.save('../models/ncf_model.h5')
print("Model saved to models/ncf_model.h5")